# 04 — Train a CLS-preserving residual GraphSAGE model

This notebook restores the original residual GraphSAGE experiment.

The graph branch is:

`DINOv2 patch embeddings → cross-image patch graph → GraphSAGE → mean prototype cosine logits`

The final classifier is:

`final logits = frozen CLS logits + α × GraphSAGE graph logits`

`α` starts at zero, so the model initially matches the frozen CLS baseline exactly.

This notebook is distinct from the later GATv2 ablations.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source:", SRC_DIR)


In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)

The model in this notebook uses **GraphSAGE**, not GATv2.

For every query–candidate graph:

1. GraphSAGE refines the patch-node representations.
2. Refined patches are mean-pooled into query/support image embeddings.
3. Support embeddings are averaged into a class prototype.
4. A cosine graph logit is produced.
5. The graph logit is used as a learned residual correction to the frozen CLS logit.

So the final score is:

`CLS logits + α × GraphSAGE logits`.


In [ ]:
from cross_image_glot.models import (
    PatchGraphSAGEEncoder,
    MeanPrototypeCosineReadout,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)

from cross_image_glot.training import (
    evaluate_residual_dataset,
    load_training_checkpoint,
    make_checkpoint,
    save_checkpoint_atomic,
    save_history,
    train_residual_epoch,
)

from cross_image_glot.storage import restore_feature_splits
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder

restore_feature_splits(
    ["train", "val"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

train_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "train",
    max_cached_shards=6,
)

val_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "val",
    max_cached_shards=6,
)

# IMPORTANT: this is the original residual GraphSAGE config.
config = json.loads(
    Path("configs/residual_5shot.json").read_text()
)

train_episodes = FewShotFeatureEpisodeDataset(
    train_features,
    config["n_way"],
    config["k_shot"],
    config["train_queries_per_class"],
    num_episodes=1000,
    seed=config["train_seed"],
    vary_by_epoch=True,
)

val_episodes = FewShotFeatureEpisodeDataset(
    val_features,
    config["n_way"],
    config["k_shot"],
    config["eval_queries_per_class"],
    num_episodes=600,
    seed=config["val_seed"],
    vary_by_epoch=False,
)

graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(train_features.metadata["grid_size"]),
    top_k=config["top_k"],
    min_similarity=None,
    graph_dtype=torch.float32,
    similarity_device=device,
)

# Original graph encoder: GraphSAGE.
encoder = PatchGraphSAGEEncoder(
    input_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    dropout=config["dropout"],
)

readout = MeanPrototypeCosineReadout(
    temperature=config["graph_temperature"],
    learnable_temperature=False,
)

graph_matcher = CrossImageGraphMatcher(
    encoder=encoder,
    readout=readout,
)

# Residual wrapper: frozen CLS logits + alpha * GraphSAGE logits.
model = BaselinePreservingResidualMatcher(
    graph_matcher,
    config["initial_residual_scale"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

print("Model: residual GraphSAGE")
print("Experiment:", config["experiment_name"])
print("Initial residual scale:", float(model.residual_scale.detach().cpu()))


In [ ]:
checkpoint_dir = paths.drive_checkpoint_dir / config["experiment_name"]
result_dir = paths.drive_results_dir / config["experiment_name"]
checkpoint_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)
latest, best = checkpoint_dir / "latest.pt", checkpoint_dir / "best.pt"
history, start_epoch = [], 0
best_accuracy, without_improvement = float("-inf"), 0
RESUME = True
if RESUME and latest.exists():
    state = load_training_checkpoint(latest, model, optimizer, device)
    start_epoch = state["epoch"] + 1
    best_accuracy = state["best_validation_accuracy"]
    without_improvement = state["epochs_without_improvement"]
    history = state.get("history", [])

In [ ]:
for epoch in range(start_epoch, config["num_epochs"]):
    print(f"\nEpoch {epoch + 1}/{config['num_epochs']}")
    train_metrics = train_residual_epoch(
        model, optimizer, graph_builder, train_episodes, device, epoch,
        config["train_episodes_per_epoch"], config["graph_microbatch_size"],
        config["cls_temperature"], log_interval=10,
    )
    val_metrics = evaluate_residual_dataset(
        model, graph_builder, val_episodes, device,
        config["validation_episodes_per_epoch"], config["graph_microbatch_size"],
        config["cls_temperature"], log_interval=5,
    )
    record = {
        "epoch": epoch,
        "train_loss": train_metrics.loss,
        "train_accuracy": train_metrics.accuracy,
        "validation_loss": val_metrics.loss,
        "validation_accuracy": val_metrics.accuracy,
        "residual_scale": float(model.residual_scale.detach().cpu()),
    }
    history.append(record)
    improved = val_metrics.accuracy > best_accuracy
    if improved:
        best_accuracy, without_improvement = val_metrics.accuracy, 0
    else:
        without_improvement += 1
    state = make_checkpoint(model, optimizer, epoch, best_accuracy, without_improvement, history, config)
    save_checkpoint_atomic(state, latest)
    if improved:
        save_checkpoint_atomic(state, best)
    save_history(history, result_dir)
    print(record, "best=", best_accuracy)
    if without_improvement >= config["early_stopping_patience"]:
        print("Early stopping.")
        break

In [ ]:
state = torch.load(
    best,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(state["model_state_dict"])
model.to(device)
model.eval()

final_metrics = evaluate_residual_dataset(
    model,
    graph_builder,
    val_episodes,
    device,
    config["final_validation_episodes"],
    config["graph_microbatch_size"],
    config["cls_temperature"],
    log_interval=10,
)

from cross_image_glot.storage import atomic_json_save

atomic_json_save(
    {
        **final_metrics.to_dict(),
        "model_kind": "residual_graphsage",
        "residual_scale": float(model.residual_scale.detach().cpu()),
    },
    result_dir / "validation_metrics.json",
)

print("Residual GraphSAGE validation:", final_metrics)
print("Residual scale:", model.residual_scale.item())
